## HEA4 DOSのデータ生成

HEA4の4つの構成元素からDOSが作られます。
この４つの構成元素から説明変数を生成しますが、
[Z1,Z2,Z3,Z4]というリストと[Z2,Z2,Z4,Z3]などの順序を変えたリストは同じ物質を表します。
たとえば、平均値と標準偏差に直すことで、順序依存性をなくすことができます。
ここではその最も簡単な方法にょり説明変数生成を行います。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pymatgen.core import Element
from copy import deepcopy
import seaborn as sns
import os
    
pd.set_option("display.max_rows", 1000)

In [ ]:
NDIV = 10  # digitizeする分割数。

元データ
ref. https: // doi.org/10.5281/zenodo.5105047.


In [ ]:
!head ../data/hea4_physprop.csv


In [ ]:
def get_data():
    """get data.

    Returns:
        pd.DataFrame: data,
        [str]: element symbols,
        [str]: target names.
        [str]: meta data names.
    """
    element_labels = []
    for i in range(4):
        element_labels.append("element{}".format(i+1))
    target_names = ['M', 'TC', 'R', ]
    meta_names = ['heakey', ]
    filepath = "../data/hea4_physprop.csv"

    dfraw = pd.read_csv(filepath)
    dfraw = dfraw.dropna().reset_index(drop=True)  # NaNデータを除く。
    return dfraw, element_labels, target_names, meta_names


g_dfraw, g_element_labels, g_target_names, g_meta_names = get_data()
g_dfraw


In [ ]:
def get_elements(dfraw, element_labels):
    """add elements to DataFrame.

    Args:
        dfraw (pd.DataFrame): data.
        element_labels ([str]]): element name column name.

    Returns:
        pd.DataFrame: data.
    """
    dfnew = pd.DataFrame()
    elements = []
    for name in dfraw[element_labels].values:
        s = ","+",".join(name.tolist())+","
        elements.append(s)
    dfnew["elements"] = elements
    return dfnew


g_df_elements = get_elements(g_dfraw, g_element_labels)
g_df_elements


In [ ]:
for _name in g_target_names:
    g_dfraw.hist(_name, bins=10)


M, TCは0（つまり非磁性解）の頻度が大きい。
頻度が特に大きいと解析に不都合なのであとで除く。

あとで結合するためにdflistに保存する。

```
pip install progressbar2
```

In [ ]:
from progressbar import ProgressBar
def add_number_grouprow(df, element_labels):
    """add explnatory variables related to group and rows of periodic table to dataframe.
    'group' and 'row' columns will be added.

    Args:
        df (pd.DataFrame): data.
        element_labels (str): element column label names.

    Returns:
        pd.DataFrame: dataframe where new columns are added.
    """
    df = deepcopy(df)

    group = {}
    groupnamelist = list(range(3, 16))
    for i in groupnamelist:
        group[i] = []
    row = {}
    rownamelist = list(range(3, 7))
    for i in rownamelist:
        row[i] = []

    result = {"group": [], "row": []}

    pbar = ProgressBar(max_value=df.shape[0])
    for i, elms in enumerate(df[element_labels].values):
        if i % 1000 == 0:
            pbar.update(i+1)
        _grouplist = []
        _rowlist = []
        for name in elms:
            elm = Element[name]
            _grouplist.append(elm.group)
            _rowlist.append(elm.row)

        grouparray = np.zeros(len(groupnamelist))
        rowarray = np.zeros(len(rownamelist))

        for group in _grouplist:
            igroup = groupnamelist.index(group)
            grouparray[igroup] += 1
        for row in _rowlist:
            irow = rownamelist.index(row)
            rowarray[irow] += 1

        result["group"].append(grouparray.astype(int).tolist())
        result["row"].append(rowarray.astype(int).tolist())

    _groupnamelist = ["n_group"+str(x) for x in groupnamelist]
    _rownamelist = ["n_row"+str(x) for x in rownamelist]
    df_group = pd.DataFrame(result["group"], columns=_groupnamelist)
    df_row = pd.DataFrame(result["row"], columns=_rownamelist)

    dfnew = pd.concat([df_group, df_row], axis=1)
    return dfnew


g_df_number_grouprow = add_number_grouprow(g_dfraw, g_element_labels, )


In [ ]:
def add_operations(df, element_labels,
                   ops=[np.mean, np.std],
                   op_names=["mean", "std"]):
    """take operations and add columns to the dataframe.

    'group_mean', 'group_sd', 'row_mean', 'row_std' are added.

    Args:
        df (pd.DataFrame): data.
        element_labels ([type]): [description]
        ops (list, optional): [description]. Defaults to [np.mean, np.std].
        op_names (list, optional): [description]. Defaults to ["mean", "std"].

    Returns:
        pd.DataFrame: data where new columns are added.
    """
    dfnew = pd.DataFrame()
    group = {}
    row = {}
    for name in op_names:
        group[name] = []
        row[name] = []

    for i, elms in enumerate(df[element_labels].values):
        grouplist = []
        rowlist = []
        for name in elms:
            elm = Element[name]
            grouplist.append(elm.group)
            rowlist.append(elm.row)

        for name, op in zip(op_names, ops):
            group[name].append(op(grouplist))
            row[name].append(op(rowlist))

    for name in op_names:
        feature = "group"
        newname = "{}_{}".format(feature, name)
        dfnew[newname] = group[name]
    for name in op_names:
        feature = "row"
        newname = "{}_{}".format(feature, name)
        dfnew[newname] = row[name]
    return dfnew


g_df_mean_std = add_operations(g_dfraw, g_element_labels, )
g_df_mean_std


In [ ]:
def add_diginized_values(df, process_labels, ndiv=5):
    """add digitized values to the dataframe.
    
    '{column name}_id' columns are added. Their values are in [0,ndiv-1].

    Args:
        df (pd.DataFrame): data.
        process_labels ([str]): a list of column names to process.
        ndiv (int, optional): the number of divisions. Defaults to 5.

    Returns:
        [type]: [description]
    """
    df = df.copy()
    dfnew = pd.DataFrame()
    for name in process_labels:
        TC = df[name].values
        eps = 1e-5
        TCmin = TC.min()-eps
        TCmax = TC.max()+eps
        bins = np.linspace(TCmin, TCmax, ndiv+1)
        id_ = np.digitize(df[name].values, bins)
        nameid = "{}_id".format(name)
        dfnew[nameid] = id_
        print("added columns", nameid)
    return dfnew


g_digitize_labels = deepcopy(g_target_names)
g_digitize_labels.extend(['group_mean', 'group_std', 'row_mean', 'row_std'])

g_df_to_digitize = pd.concat([g_df_mean_std, g_dfraw], axis=1)
g_df_digitized_id = add_diginized_values(
    g_df_to_digitize, g_digitize_labels, ndiv=NDIV)
g_df_digitized_id


In [ ]:
dflist = [g_dfraw]
dflist.append(g_df_elements)
dflist.append(g_df_number_grouprow)
dflist.append(g_df_mean_std)
dflist.append(g_df_digitized_id)
df_processed = pd.concat(dflist, axis=1)
# DataFrameを結合
df_processed


In [ ]:
# 加工済みデータの保存
filepath = os.path.join("../data_calculated/hea4_phys.csv")
df_processed.to_csv(filepath, index=False)


In [ ]:
# 計算条件の書き込み
import json
filepath = os.path.join("../data_calculated/hea4_phys_condition.json")
with open(filepath, "w") as f:
    json.dump({"NDIV": NDIV}, f)


In [ ]:
print("all done")
